In [ ]:
from flask import Flask, render_template_string, request, redirect, url_for, session
import random

app = Flask(__name__)
app.secret_key = 'your_secret_key_here'

# Simulated in-memory "database"
users = []
predictions = []
feedbacks = []

# Common CSS Styles
common_styles = """
<style>
    body {
        margin: 0;
        padding: 0;
        display: flex;
        justify-content: center;
        align-items: center;
        min-height: 100vh;
        background: linear-gradient(rgba(0, 0, 0, 0.7), rgba(0, 0, 0, 0.7)),
                    url('https://images.unsplash.com/photo-1515694346937-94d85e41e6f0?ixlib=rb-1.2.1&auto=format&fit=crop&w=1350&q=80') no-repeat center center/cover;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        color: white;
        text-align: center;
    }
    .container {
        background: rgba(0, 0, 0, 0.6);
        padding: 2rem;
        border-radius: 10px;
        box-shadow: 0 0 20px rgba(0, 0, 0, 0.5);
        width: 90%;
        max-width: 500px;
        margin: 2rem;
    }
    h1 {
        font-size: 2rem;
        margin-bottom: 1.5rem;
        text-transform: uppercase;
    }
    .btn {
        background: #ffcc00;
        color: black;
        border: none;
        padding: 0.8rem 1.5rem;
        font-size: 1rem;
        cursor: pointer;
        border-radius: 5px;
        text-decoration: none;
        display: inline-block;
        transition: 0.3s;
        margin: 0.5rem;
        font-weight: bold;
    }
    .btn:hover {
        background: #ffaa00;
        transform: translateY(-2px);
    }
    input, textarea, select {
        width: 100%;
        padding: 0.8rem;
        margin: 0.5rem 0;
        border-radius: 5px;
        border: none;
        box-sizing: border-box;
    }
    .nav-links {
        margin-top: 1.5rem;
        display: flex;
        justify-content: center;
        flex-wrap: wrap;
    }
</style>
"""

# HTML Pages
welcome_page = f"""<!DOCTYPE html><html><head><title>Welcome</title>{common_styles}</head><body>
<div class="container">
    <h1>Welcome to Rainfall Prediction</h1>
    <p>An AI-powered solution for accurate precipitation forecasting</p>
    <div class="nav-links">
        <a href="/about" class="btn">Learn More</a>
        <a href="/register" class="btn">Get Started</a>
        <a href="/contact" class="btn">Contact</a>
    </div>
</div></body></html>"""

about_page = f"""<!DOCTYPE html><html><head><title>About</title>{common_styles}</head><body>
<div class="container">
    <h1>About Our System</h1>
    <p>Our rainfall prediction system uses advanced logic to forecast precipitation with high accuracy.</p>
    <ul style="text-align: left; margin-left: 1.5rem;">
        <li>Real-time weather data analysis</li>
        <li>Historical pattern recognition</li>
        <li>User-friendly interface</li>
    </ul>
    <div class="nav-links">
        <a href="/" class="btn">Home</a>
        <a href="/register" class="btn">Register</a>
    </div>
</div></body></html>"""

contact_page = f"""<!DOCTYPE html><html><head><title>Contact</title>{common_styles}</head><body>
<div class="container">
    <h1>Contact Us</h1>
    <p>If you have questions, reach out to us via email:</p>
    <p><strong>hajay8727@gmail.com</strong></p>
    <div class="nav-links">
        <a href="/" class="btn">Back to Home</a>
    </div>
</div></body></html>"""

register_page = f"""<!DOCTYPE html><html><head><title>Register</title>{common_styles}</head><body>
<div class="container">
    <h1>Create Account</h1>
    <form action="/register" method="POST">
        <input type="text" name="username" placeholder="Username" required>
        <input type="email" name="email" placeholder="Email" required>
        <input type="password" name="password" placeholder="Password" required>
        <button type="submit" class="btn">Register</button>
    </form>
    <p>Already have an account? <a href="/login" style="color: #ffcc00;">Login here</a></p>
</div></body></html>"""

login_page = f"""<!DOCTYPE html><html><head><title>Login</title>{common_styles}</head><body>
<div class="container">
    <h1>Login</h1>
    <form action="/login" method="POST">
        <input type="text" name="username" placeholder="Username" required>
        <input type="password" name="password" placeholder="Password" required>
        <button type="submit" class="btn">Login</button>
    </form>
    <p>Don't have an account? <a href="/register" style="color: #ffcc00;">Register here</a></p>
</div></body></html>"""

input_page = f"""<!DOCTYPE html><html><head><title>Input</title>{common_styles}</head><body>
<div class="container">
    <h1>Enter Weather Data</h1>
    <form action="/predict" method="POST">
        <input type="text" name="location" placeholder="Location" required>
        <input type="number" step="0.1" name="temperature" placeholder="Temperature (°C)" required>
        <input type="number" step="0.1" name="humidity" placeholder="Humidity (%)" required>
        <input type="number" step="0.1" name="wind_speed" placeholder="Wind Speed (km/h)" required>
        <select name="season" required>
            <option value="">Select Season</option>
            <option value="winter">Winter</option>
            <option value="spring">Spring</option>
            <option value="summer">Summer</option>
            <option value="autumn">Autumn</option>
        </select>
        <button type="submit" class="btn">Predict Rainfall</button>
    </form>
</div></body></html>"""

def get_result_page(prediction, confidence):
    return f"""<!DOCTYPE html><html><head><title>Result</title>{common_styles}</head><body>
<div class="container">
    <h1>Prediction Result</h1>
    <p>Rainfall prediction: <strong>{prediction}</strong></p>
    <p>Confidence: <strong>{confidence}%</strong></p>
    <div class="nav-links">
        <a href="/input" class="btn">New Prediction</a>
        <a href="/feedback" class="btn">Provide Feedback</a>
        <a href="/dashboard" class="btn">Dashboard</a>
    </div>
</div></body></html>"""

feedback_page = f"""<!DOCTYPE html><html><head><title>Feedback</title>{common_styles}</head><body>
<div class="container">
    <h1>We Value Your Feedback</h1>
    <form action="/submit_feedback" method="POST">
        <textarea name="feedback" placeholder="Your feedback helps us improve..." rows="5" required></textarea>
        <div style="margin: 1rem 0;">
            <label>Rating:</label>
            <select name="rating" required>
                <option value="">Select rating</option>
                <option value="5">Excellent</option>
                <option value="4">Good</option>
                <option value="3">Average</option>
                <option value="2">Poor</option>
                <option value="1">Terrible</option>
            </select>
        </div>
        <button type="submit" class="btn">Submit Feedback</button>
    </form>
</div></body></html>"""

thankyou_page = f"""<!DOCTYPE html><html><head><title>Thank You</title>{common_styles}</head><body>
<div class="container">
    <h1>Thank You!</h1>
    <p>We appreciate your feedback and will use it to improve our service.</p>
    <div class="nav-links">
        <a href="/" class="btn">Home</a>
        <a href="/dashboard" class="btn">Dashboard</a>
    </div>
</div></body></html>"""

dashboard_page = f"""<!DOCTYPE html><html><head><title>Dashboard</title>{common_styles}</head><body>
<div class="container">
    <h1>Your Dashboard</h1>
    <div class="nav-links">
        <a href="/input" class="btn">New Prediction</a>
        <a href="/admin_login" class="btn">Go to Admin Panel</a>
        <a href="/" class="btn">Home</a>
        <a href="/logout" class="btn">Logout</a>
    </div>
</div></body></html>"""

admin_login_page = f"""<!DOCTYPE html><html><head><title>Admin Login</title>{common_styles}</head><body>
<div class="container">
    <h1>Admin Login</h1>
    <form method="POST">
        <input type="text" name="adminname" placeholder="Admin Name" required>
        <input type="password" name="adminpass" placeholder="Admin Password" required>
        <button type="submit" class="btn">Login</button>
    </form>
</div></body></html>"""

# Flask Routes
@app.route('/')
def welcome():
    return render_template_string(welcome_page)

@app.route('/about')
def about():
    return render_template_string(about_page)

@app.route('/contact')
def contact():
    return render_template_string(contact_page)

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        users.append({
            'username': request.form['username'],
            'email': request.form['email'],
            'password': request.form['password']
        })
        return redirect(url_for('login'))
    return render_template_string(register_page)

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        for user in users:
            if user['username'] == request.form['username'] and user['password'] == request.form['password']:
                session['user'] = user['username']
                return redirect(url_for('dashboard'))
        return "Invalid credentials. Please try again."
    return render_template_string(login_page)

@app.route('/logout')
def logout():
    session.pop('user', None)
    session.pop('admin', None)
    return redirect(url_for('welcome'))

@app.route('/dashboard')
def dashboard():
    if 'user' not in session:
        return redirect(url_for('login'))
    return render_template_string(dashboard_page)

@app.route('/input')
def input_data():
    if 'user' not in session:
        return redirect(url_for('login'))
    return render_template_string(input_page)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.form
    temperature = float(data['temperature'])
    humidity = float(data['humidity'])
    prediction = "Heavy Rain" if humidity > 70 and temperature < 30 else "Moderate Rain" if humidity > 50 else "Light Rain"
    confidence = random.randint(80, 95) if prediction == "Heavy Rain" else random.randint(60, 80) if prediction == "Moderate Rain" else random.randint(40, 60)
    predictions.append({**data, 'prediction': prediction, 'confidence': confidence})
    return render_template_string(get_result_page(prediction, confidence))

@app.route('/feedback')
def feedback():
    return render_template_string(feedback_page)

@app.route('/submit_feedback', methods=['POST'])
def submit_feedback():
    feedbacks.append({
        'feedback': request.form['feedback'],
        'rating': request.form['rating']
    })
    return redirect(url_for('thankyou'))

@app.route('/thankyou')
def thankyou():
    return render_template_string(thankyou_page)

@app.route('/admin_login', methods=['GET', 'POST'])
def admin_login():
    if request.method == 'POST':
        if request.form['adminname'] == 'ajay' and request.form['adminpass'] == '123':
            session['admin'] = True
            return redirect(url_for('admin_panel'))
        return "<h2 style='color: red;'>Invalid name or password</h2>"
    return render_template_string(admin_login_page)

@app.route('/admin', methods=['GET', 'POST'])
def admin_panel():
    if 'admin' not in session:
        return redirect(url_for('admin_login'))

    if request.method == 'POST':
        if request.form['action'] == 'add':
            predictions.append({
                'location': request.form['location'],
                'temperature': float(request.form['temperature']),
                'humidity': float(request.form['humidity']),
                'wind_speed': float(request.form['wind_speed']),
                'season': request.form['season'],
                'prediction': request.form['prediction'],
                'confidence': int(request.form['confidence'])
            })
        elif request.form['action'] == 'delete':
            idx = int(request.form['index'])
            if 0 <= idx < len(predictions):
                predictions.pop(idx)

    items = "".join(
        f"<li>{i + 1}. {p['location']} - {p['prediction']} ({p['confidence']}%) "
        f"<form method='POST' style='display:inline'><input type='hidden' name='index' value='{i}'><input type='hidden' name='action' value='delete'><button class='btn'>Delete</button></form></li>"
        for i, p in enumerate(predictions)
    )

    add_form = """
    <form method="POST">
        <input name="location" placeholder="Location" required>
        <input name="temperature" placeholder="Temperature" type="number" required>
        <input name="humidity" placeholder="Humidity" type="number" required>
        <input name="wind_speed" placeholder="Wind Speed" type="number" required>
        <input name="season" placeholder="Season" required>
        <input name="prediction" placeholder="Prediction" required>
        <input name="confidence" placeholder="Confidence" type="number" required>
        <input type="hidden" name="action" value="add">
        <button class="btn">Add Data</button>
    </form>
    """

    return f"""<!DOCTYPE html><html><head><title>Admin Panel</title>{common_styles}</head><body>
    <div class="container">
        <h1>Admin Panel</h1>
        <h2>Prediction Data</h2>
        <ul>{items or '<li>No predictions yet.</li>'}</ul>
        <h2>Add Prediction</h2>
        {add_form}
        <div class="nav-links">
            <a href="/dashboard" class="btn">Back to Dashboard</a>
        </div>
    </div></body></html>"""

if __name__ == '__main__':
    app.run(debug=True)